# Evaluating RAG Performance

This notebook builds RAG evaluation metrics from scratch using a toy example. You will compute Precision@K, Recall@K, Mean Reciprocal Rank (MRR), and Normalized Discounted Cumulative Gain (NDCG) step by step so you can see exactly how each metric works and what it tells you about retrieval quality.

## Setup

Only numpy is needed for the math. No external retrieval frameworks required.

In [ ]:
%pip install numpy --upgrade

In [1]:
import numpy as np

## Our Toy Knowledge Base

Imagine you are building a RAG system for a cooking assistant. You have 10 documents in your vector store. Throughout this notebook, you will evaluate how well a retriever finds the right documents for different queries.

In [3]:
# Our knowledge base: 10 cooking-related documents
documents = {
    "D0": "Preheat the oven to 375°F for baking chocolate chip cookies.",
    "D1": "To make pasta al dente, boil for 8-10 minutes in salted water.",
    "D2": "Chocolate ganache requires equal parts chocolate and heavy cream.",
    "D3": "Knead bread dough for 10 minutes until smooth and elastic.",
    "D4": "Season cast iron pans with vegetable oil at 450°F for one hour.",
    "D5": "Resting meat after cooking lets juices redistribute for 5-10 minutes.",
    "D6": "Caramelize onions over low heat for 30-45 minutes, stirring occasionally.",
    "D7": "Use room temperature eggs and butter for fluffier cake batter.",
    "D8": "Blanch vegetables in boiling water then ice bath to preserve color.",
    "D9": "Temper chocolate by heating to 115°F, cooling to 82°F, then reheating to 90°F.",
}

print(f"Knowledge base: {len(documents)} documents")
for doc_id, text in documents.items():
    print(f"  {doc_id}: {text}")

Knowledge base: 10 documents
  D0: Preheat the oven to 375°F for baking chocolate chip cookies.
  D1: To make pasta al dente, boil for 8-10 minutes in salted water.
  D2: Chocolate ganache requires equal parts chocolate and heavy cream.
  D3: Knead bread dough for 10 minutes until smooth and elastic.
  D4: Season cast iron pans with vegetable oil at 450°F for one hour.
  D5: Resting meat after cooking lets juices redistribute for 5-10 minutes.
  D6: Caramelize onions over low heat for 30-45 minutes, stirring occasionally.
  D7: Use room temperature eggs and butter for fluffier cake batter.
  D8: Blanch vegetables in boiling water then ice bath to preserve color.
  D9: Temper chocolate by heating to 115°F, cooling to 82°F, then reheating to 90°F.


## Ground Truth: What *Should* Be Retrieved?

For each query, you need to define which documents are actually relevant. This is the **ground truth**, the labels a human would assign. Without ground truth, there is no way to measure retrieval quality.

Two levels of relevance are used here:

**2** = Highly relevant (directly answers the query)

**1** = Somewhat relevant (related but not a direct answer)

**0** = Not relevant (default for unlisted documents)

In [4]:
# Ground truth: for each query, which documents are relevant and how relevant?
queries_and_relevance = {
    "How do I bake cookies?": {
        "D0": 2,  # Directly about baking cookies
        "D7": 1,  # Room temp ingredients, useful for baking
        "D2": 1,  # Chocolate ganache, could be a topping
    },
    "How should I cook pasta?": {
        "D1": 2,  # Directly about cooking pasta
        "D8": 1,  # Blanching, related cooking technique
    },
    "Tell me about working with chocolate": {
        "D2": 2,  # Chocolate ganache
        "D9": 2,  # Tempering chocolate
        "D0": 1,  # Chocolate chip cookies
    },
}

for query, rels in queries_and_relevance.items():
    print(f"\nQuery: '{query}'")
    for doc_id, score in rels.items():
        label = "highly relevant" if score == 2 else "somewhat relevant"
        print(f"  {doc_id} ({label}): {documents[doc_id][:60]}...")


Query: 'How do I bake cookies?'
  D0 (highly relevant): Preheat the oven to 375°F for baking chocolate chip cookies....
  D7 (somewhat relevant): Use room temperature eggs and butter for fluffier cake batte...
  D2 (somewhat relevant): Chocolate ganache requires equal parts chocolate and heavy c...

Query: 'How should I cook pasta?'
  D1 (highly relevant): To make pasta al dente, boil for 8-10 minutes in salted wate...
  D8 (somewhat relevant): Blanch vegetables in boiling water then ice bath to preserve...

Query: 'Tell me about working with chocolate'
  D2 (highly relevant): Chocolate ganache requires equal parts chocolate and heavy c...
  D9 (highly relevant): Temper chocolate by heating to 115°F, cooling to 82°F, then ...
  D0 (somewhat relevant): Preheat the oven to 375°F for baking chocolate chip cookies....


## Simulated Retriever Results

In a real RAG system, these results would come from your vector store (e.g. Supabase, Pinecone, Chroma). Here they are simulated, including some deliberate mistakes, so you can observe how the metrics respond to imperfect retrieval.

In [5]:
# Simulated retriever output: ordered list of (doc_id, similarity_score)
# The retriever returns its top 5 documents for each query
retriever_results = {
    "How do I bake cookies?": [
        ("D0", 0.95),  # Highly relevant (rank 1)
        ("D3", 0.82),  # Not relevant: bread, not cookies
        ("D7", 0.78),  # Somewhat relevant (rank 3)
        ("D4", 0.71),  # Not relevant: cast iron
        ("D2", 0.65),  # Somewhat relevant (rank 5)
    ],
    "How should I cook pasta?": [
        ("D6", 0.88),  # Not relevant: onions
        ("D1", 0.85),  # Highly relevant (rank 2, not rank 1!)
        ("D8", 0.72),  # Somewhat relevant (rank 3)
        ("D5", 0.68),  # Not relevant: resting meat
        ("D4", 0.61),  # Not relevant: cast iron
    ],
    "Tell me about working with chocolate": [
        ("D9", 0.93),  # Highly relevant (rank 1)
        ("D2", 0.91),  # Highly relevant (rank 2)
        ("D0", 0.76),  # Somewhat relevant (rank 3)
        ("D7", 0.69),  # Not relevant
        ("D3", 0.55),  # Not relevant
    ],
}

for query, results in retriever_results.items():
    relevance = queries_and_relevance[query]
    print(f"\nQuery: '{query}'")
    for rank, (doc_id, sim) in enumerate(results, 1):
        rel = relevance.get(doc_id, 0)
        marker = "relevant" if rel > 0 else "not relevant"
        print(f"  Rank {rank}: {doc_id} (sim={sim:.2f}) [{marker}, rel={rel}]")


Query: 'How do I bake cookies?'
  Rank 1: D0 (sim=0.95) [relevant, rel=2]
  Rank 2: D3 (sim=0.82) [not relevant, rel=0]
  Rank 3: D7 (sim=0.78) [relevant, rel=1]
  Rank 4: D4 (sim=0.71) [not relevant, rel=0]
  Rank 5: D2 (sim=0.65) [relevant, rel=1]

Query: 'How should I cook pasta?'
  Rank 1: D6 (sim=0.88) [not relevant, rel=0]
  Rank 2: D1 (sim=0.85) [relevant, rel=2]
  Rank 3: D8 (sim=0.72) [relevant, rel=1]
  Rank 4: D5 (sim=0.68) [not relevant, rel=0]
  Rank 5: D4 (sim=0.61) [not relevant, rel=0]

Query: 'Tell me about working with chocolate'
  Rank 1: D9 (sim=0.93) [relevant, rel=2]
  Rank 2: D2 (sim=0.91) [relevant, rel=2]
  Rank 3: D0 (sim=0.76) [relevant, rel=1]
  Rank 4: D7 (sim=0.69) [not relevant, rel=0]
  Rank 5: D3 (sim=0.55) [not relevant, rel=0]


## Metric 1: Precision@K and Recall@K

The simplest retrieval metrics. They answer two questions:

**Precision@K**: Of the K documents retrieved, how many were relevant?

**Recall@K**: Of all the relevant documents that exist, how many appeared in the top K?

$$\text{Precision@K} = \frac{\text{relevant documents in top K}}{K}$$

$$\text{Recall@K} = \frac{\text{relevant documents in top K}}{\text{total relevant documents}}$$

In [6]:
def precision_at_k(retrieved: list[str], relevant: dict[str, int], k: int) -> float:
    """What fraction of the top-K retrieved docs are relevant?"""
    top_k = retrieved[:k]
    relevant_in_top_k = sum(1 for doc_id in top_k if relevant.get(doc_id, 0) > 0)
    return relevant_in_top_k / k


def recall_at_k(retrieved: list[str], relevant: dict[str, int], k: int) -> float:
    """What fraction of all relevant docs appear in the top K?"""
    top_k = retrieved[:k]
    total_relevant = sum(1 for v in relevant.values() if v > 0)
    if total_relevant == 0:
        return 0.0
    relevant_in_top_k = sum(1 for doc_id in top_k if relevant.get(doc_id, 0) > 0)
    return relevant_in_top_k / total_relevant

In [7]:
# Walk through the cookie query step by step
query = "How do I bake cookies?"
retrieved = [doc_id for doc_id, _ in retriever_results[query]]
relevant = queries_and_relevance[query]

print(f"Query: '{query}'")
print(f"Retrieved docs: {retrieved}")
print(f"Relevant docs:  {list(relevant.keys())} (total: {len(relevant)})")
print()

# Watch how precision and recall change as K grows
print(f"{'K':>3} | {'Top-K Retrieved':^30} | {'Relevant?':^20} | {'P@K':>6} | {'R@K':>6}")
print("=" * 85)

for k in range(1, 6):
    top_k = retrieved[:k]
    relevance_marks = ["Y" if relevant.get(d, 0) > 0 else "N" for d in top_k]
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    print(f"{k:>3} | {str(top_k):^30} | {str(relevance_marks):^20} | {p:>6.3f} | {r:>6.3f}")

Query: 'How do I bake cookies?'
Retrieved docs: ['D0', 'D3', 'D7', 'D4', 'D2']
Relevant docs:  ['D0', 'D7', 'D2'] (total: 3)

  K |        Top-K Retrieved         |      Relevant?       |    P@K |    R@K
  1 |             ['D0']             |        ['Y']         |  1.000 |  0.333
  2 |          ['D0', 'D3']          |      ['Y', 'N']      |  0.500 |  0.333
  3 |       ['D0', 'D3', 'D7']       |   ['Y', 'N', 'Y']    |  0.667 |  0.667
  4 |    ['D0', 'D3', 'D7', 'D4']    | ['Y', 'N', 'Y', 'N'] |  0.500 |  0.667
  5 | ['D0', 'D3', 'D7', 'D4', 'D2'] | ['Y', 'N', 'Y', 'N', 'Y'] |  0.600 |  1.000


Notice the **precision/recall tradeoff**: as K grows, recall increases (you find more relevant docs) but precision can drop (you also include more irrelevant ones). For RAG with LLMs, K=3 to K=5 is common since you want enough context without overwhelming the model.

In [8]:
# Compute P@3 and R@3 across all queries
print("Precision@3 and Recall@3 across all queries:\n")
p3_scores = []
r3_scores = []

for query, results in retriever_results.items():
    retrieved = [doc_id for doc_id, _ in results]
    relevant = queries_and_relevance[query]
    p3 = precision_at_k(retrieved, relevant, 3)
    r3 = recall_at_k(retrieved, relevant, 3)
    p3_scores.append(p3)
    r3_scores.append(r3)
    print(f"  '{query}'")
    print(f"    P@3 = {p3:.3f}, R@3 = {r3:.3f}")

print(f"\nAverage P@3 = {np.mean(p3_scores):.3f}")
print(f"Average R@3 = {np.mean(r3_scores):.3f}")

Precision@3 and Recall@3 across all queries:

  'How do I bake cookies?'
    P@3 = 0.667, R@3 = 0.667
  'How should I cook pasta?'
    P@3 = 0.667, R@3 = 1.000
  'Tell me about working with chocolate'
    P@3 = 1.000, R@3 = 1.000

Average P@3 = 0.778
Average R@3 = 0.889


## Metric 2: Mean Reciprocal Rank (MRR)

MRR answers: **how quickly does the retriever find the first relevant document?**

For each query, find the rank of the **first** relevant document and take the reciprocal:

$$\text{RR}(q) = \frac{1}{\text{rank of first relevant doc}}$$

$$\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \text{RR}(q)$$

MRR = 1.0 means the first result is always relevant. MRR = 0.5 means the first relevant result is on average at rank 2.

In [9]:
def reciprocal_rank(retrieved: list[str], relevant: dict[str, int]) -> float:
    """Find the reciprocal of the rank of the first relevant document."""
    for rank, doc_id in enumerate(retrieved, 1):
        if relevant.get(doc_id, 0) > 0:
            return 1.0 / rank
    return 0.0  # No relevant docs found


# Walk through each query
print("Reciprocal Rank for each query:\n")
rr_scores = []

for query, results in retriever_results.items():
    retrieved = [doc_id for doc_id, _ in results]
    relevant = queries_and_relevance[query]
    
    # Find first relevant
    first_rel_rank = None
    for rank, doc_id in enumerate(retrieved, 1):
        if relevant.get(doc_id, 0) > 0:
            first_rel_rank = rank
            break
    
    rr = reciprocal_rank(retrieved, relevant)
    rr_scores.append(rr)
    
    print(f"  '{query}'")
    print(f"    Retrieved: {retrieved}")
    print(f"    First relevant at rank {first_rel_rank} => RR = 1/{first_rel_rank} = {rr:.3f}")
    print()

mrr = np.mean(rr_scores)
print(f"MRR = mean({[round(s, 3) for s in rr_scores]}) = {mrr:.3f}")

Reciprocal Rank for each query:

  'How do I bake cookies?'
    Retrieved: ['D0', 'D3', 'D7', 'D4', 'D2']
    First relevant at rank 1 => RR = 1/1 = 1.000

  'How should I cook pasta?'
    Retrieved: ['D6', 'D1', 'D8', 'D5', 'D4']
    First relevant at rank 2 => RR = 1/2 = 0.500

  'Tell me about working with chocolate'
    Retrieved: ['D9', 'D2', 'D0', 'D7', 'D3']
    First relevant at rank 1 => RR = 1/1 = 1.000

MRR = mean([1.0, 0.5, 1.0]) = 0.833


The pasta query pulls down the MRR because the most relevant document (D1) sits at rank 2, not rank 1. In a RAG system, this matters: if the LLM sees an irrelevant document first, it might anchor on that instead of the correct answer.

## Metric 3: Normalized Discounted Cumulative Gain (NDCG)

NDCG is the gold standard retrieval metric. Unlike Precision/Recall (which treat relevance as binary) and MRR (which only cares about the first hit), NDCG handles **graded relevance**. It knows that a "highly relevant" doc at rank 1 is much better than a "somewhat relevant" doc at rank 1.

It also **penalizes good docs appearing too late** in the ranking. You will build it piece by piece below.

### Step 1: Cumulative Gain (CG)

The simplest idea: just add up the relevance scores of all retrieved documents.

$$\text{CG@K} = \sum_{i=1}^{K} \text{rel}_i$$

**Problem:** CG does not care about order. Getting [2, 0, 1] has the same CG as [0, 1, 2], but the first ranking is clearly better for a user.

### Step 2: Discounted Cumulative Gain (DCG)

Discount (reduce) the contribution of each result based on its position. Results further down the ranking contribute less:

$$\text{DCG@K} = \sum_{i=1}^{K} \frac{\text{rel}_i}{\log_2(i + 1)}$$

The $\log_2(i+1)$ divisor means rank 1 contributes full value, rank 2 is divided by $\log_2(3) \approx 1.58$, rank 3 by $\log_2(4) = 2.0$, and so on.

### Step 3: Ideal DCG and Normalization

DCG depends on how many relevant docs exist, so you cannot compare across queries directly. Normalize by the **ideal DCG**, which is the score you would get with a perfect ranking:

$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

In [ ]:
# 10 documents
# Loop over every document
# Calculate the relevant score of the document in K (K being the number of documents you're looking at it)

In [12]:
def dcg_at_k(relevance_scores: list[int], k: int) -> float:
    """Compute Discounted Cumulative Gain at K."""
    dcg = 0.0
    for i in range(min(k, len(relevance_scores))):
        rel = relevance_scores[i]
        discount = np.log2(i + 2)  # i+2 because i is 0-indexed, formula uses 1-indexed
        dcg += rel / discount
    return dcg


def ndcg_at_k(retrieved: list[str], relevant: dict[str, int], k: int) -> float:
    """Compute NDCG@K."""
    # Get relevance scores in retrieval order
    rel_scores = [relevant.get(doc_id, 0) for doc_id in retrieved[:k]]
    
    # DCG of actual ranking
    dcg = dcg_at_k(rel_scores, k)
    
    # Ideal ranking: sort all relevant docs by relevance descending
    ideal_scores = sorted(relevant.values(), reverse=True)
    idcg = dcg_at_k(ideal_scores, k)
    
    if idcg == 0:
        return 0.0
    return dcg / idcg

In [13]:
# Walk through the chocolate query in full detail
query = "Tell me about working with chocolate"
retrieved = [doc_id for doc_id, _ in retriever_results[query]]
relevant = queries_and_relevance[query]
k = 5

print(f"Query: '{query}'")
print(f"K = {k}\n")

# Actual ranking
rel_scores = [relevant.get(doc_id, 0) for doc_id in retrieved[:k]]
print("ACTUAL ranking:")
print(f"{'Rank':>4} | {'Doc':>4} | {'Relevance':>9} | {'Discount log2(i+1)':>18} | {'Gain (rel/discount)':>20}")
print("=" * 70)

dcg = 0.0
for i, (doc_id, rel) in enumerate(zip(retrieved[:k], rel_scores)):
    discount = np.log2(i + 2)
    gain = rel / discount
    dcg += gain
    print(f"{i+1:>4} | {doc_id:>4} | {rel:>9} | {discount:>18.3f} | {gain:>20.3f}")

print(f"\nDCG@{k} = {dcg:.3f}")

# Ideal ranking
ideal_scores = sorted(relevant.values(), reverse=True)
print(f"\nIDEAL ranking (best possible): {ideal_scores}")
print(f"{'Rank':>4} | {'Relevance':>9} | {'Discount':>18} | {'Gain':>20}")
print("=" * 60)

idcg = 0.0
for i, rel in enumerate(ideal_scores):
    discount = np.log2(i + 2)
    gain = rel / discount
    idcg += gain
    print(f"{i+1:>4} | {rel:>9} | {discount:>18.3f} | {gain:>20.3f}")

print(f"\nIDCG@{k} = {idcg:.3f}")
print(f"\nNDCG@{k} = DCG / IDCG = {dcg:.3f} / {idcg:.3f} = {dcg/idcg:.3f}")

Query: 'Tell me about working with chocolate'
K = 5

ACTUAL ranking:
Rank |  Doc | Relevance | Discount log2(i+1) |  Gain (rel/discount)
   1 |   D9 |         2 |              1.000 |                2.000
   2 |   D2 |         2 |              1.585 |                1.262
   3 |   D0 |         1 |              2.000 |                0.500
   4 |   D7 |         0 |              2.322 |                0.000
   5 |   D3 |         0 |              2.585 |                0.000

DCG@5 = 3.762

IDEAL ranking (best possible): [2, 2, 1]
Rank | Relevance |           Discount |                 Gain
   1 |         2 |              1.000 |                2.000
   2 |         2 |              1.585 |                1.262
   3 |         1 |              2.000 |                0.500

IDCG@5 = 3.762

NDCG@5 = DCG / IDCG = 3.762 / 3.762 = 1.000


In [14]:
# Compute NDCG@5 for all queries
print("NDCG@5 across all queries:\n")
ndcg_scores = []

for query, results in retriever_results.items():
    retrieved = [doc_id for doc_id, _ in results]
    relevant = queries_and_relevance[query]
    score = ndcg_at_k(retrieved, relevant, 5)
    ndcg_scores.append(score)
    
    rel_scores = [relevant.get(d, 0) for d in retrieved[:5]]
    print(f"  '{query}'")
    print(f"    Relevance scores: {rel_scores} => NDCG@5 = {score:.3f}")

print(f"\nAverage NDCG@5 = {np.mean(ndcg_scores):.3f}")

NDCG@5 across all queries:

  'How do I bake cookies?'
    Relevance scores: [2, 0, 1, 0, 1] => NDCG@5 = 0.922
  'How should I cook pasta?'
    Relevance scores: [0, 2, 1, 0, 0] => NDCG@5 = 0.670
  'Tell me about working with chocolate'
    Relevance scores: [2, 2, 1, 0, 0] => NDCG@5 = 1.000

Average NDCG@5 = 0.864


The chocolate query scores highest because both highly relevant documents appear in the top 2 positions. The pasta query scores lowest because the most relevant document is at rank 2 instead of rank 1.

## Putting It All Together

Now compare all four metrics side by side and observe what each one tells you.

In [15]:
# Side-by-side comparison of all metrics
print(f"{'Query':<42} | {'P@3':>6} | {'R@3':>6} | {'MRR':>6} | {'NDCG@5':>7}")
print("=" * 80)

all_metrics = {"P@3": [], "R@3": [], "MRR": [], "NDCG@5": []}

for query, results in retriever_results.items():
    retrieved = [doc_id for doc_id, _ in results]
    relevant = queries_and_relevance[query]
    
    p3 = precision_at_k(retrieved, relevant, 3)
    r3 = recall_at_k(retrieved, relevant, 3)
    rr = reciprocal_rank(retrieved, relevant)
    ndcg = ndcg_at_k(retrieved, relevant, 5)
    
    all_metrics["P@3"].append(p3)
    all_metrics["R@3"].append(r3)
    all_metrics["MRR"].append(rr)
    all_metrics["NDCG@5"].append(ndcg)
    
    short_query = query[:40]
    print(f"{short_query:<42} | {p3:>6.3f} | {r3:>6.3f} | {rr:>6.3f} | {ndcg:>7.3f}")

print("=" * 80)
print(f"{'AVERAGE':<42} | {np.mean(all_metrics['P@3']):>6.3f} | {np.mean(all_metrics['R@3']):>6.3f} | {np.mean(all_metrics['MRR']):>6.3f} | {np.mean(all_metrics['NDCG@5']):>7.3f}")

Query                                      |    P@3 |    R@3 |    MRR |  NDCG@5
How do I bake cookies?                     |  0.667 |  0.667 |  1.000 |   0.922
How should I cook pasta?                   |  0.667 |  1.000 |  0.500 |   0.670
Tell me about working with chocolate       |  1.000 |  1.000 |  1.000 |   1.000
AVERAGE                                    |  0.778 |  0.889 |  0.833 |   0.864


## When to Use Each Metric

| Metric | Best For | Limitation |
|--------|----------|------------|
| **Precision@K** | Measuring noise in your top results | Ignores ranking order within top K |
| **Recall@K** | Ensuring you don't miss relevant docs | Does not penalize irrelevant results |
| **MRR** | "Answer on top" use cases (chatbots, Q&A) | Only cares about the first relevant hit |
| **NDCG@K** | Full ranking quality with graded relevance | Requires graded labels (more annotation work) |

## Experiment: What Happens When You Improve the Retriever?

Below is a simulated "improved" retriever. Run it and observe how the metrics respond when the ranking gets better.

In [14]:
# Improved retriever: better ranking, fewer irrelevant docs in top positions
improved_results = {
    "How do I bake cookies?": [
        ("D0", 0.97),  # Highly relevant (rank 1), same
        ("D7", 0.88),  # Somewhat relevant, moved up from rank 3
        ("D2", 0.82),  # Somewhat relevant, moved up from rank 5
        ("D3", 0.65),  # Not relevant, pushed down
        ("D4", 0.55),  # Not relevant, pushed down
    ],
    "How should I cook pasta?": [
        ("D1", 0.94),  # Highly relevant, now rank 1!
        ("D8", 0.82),  # Somewhat relevant, moved up
        ("D6", 0.71),  # Not relevant, pushed down
        ("D5", 0.62),  # Not relevant
        ("D4", 0.55),  # Not relevant
    ],
    "Tell me about working with chocolate": [
        ("D9", 0.96),  # Highly relevant (rank 1), same
        ("D2", 0.93),  # Highly relevant (rank 2), same
        ("D0", 0.81),  # Somewhat relevant (rank 3), same
        ("D7", 0.62),  # Not relevant
        ("D3", 0.50),  # Not relevant
    ],
}

print(f"{'':40} | {'  Original  ':^27} | {'  Improved  ':^27}")
print(f"{'Query':<40} | {'P@3':>6} {'R@3':>6} {'MRR':>6} {'NDCG':>6} | {'P@3':>6} {'R@3':>6} {'MRR':>6} {'NDCG':>6}")
print("=" * 100)

for query in retriever_results:
    relevant = queries_and_relevance[query]
    
    # Original
    orig_ret = [d for d, _ in retriever_results[query]]
    op3 = precision_at_k(orig_ret, relevant, 3)
    or3 = recall_at_k(orig_ret, relevant, 3)
    orr = reciprocal_rank(orig_ret, relevant)
    on = ndcg_at_k(orig_ret, relevant, 5)
    
    # Improved
    imp_ret = [d for d, _ in improved_results[query]]
    ip3 = precision_at_k(imp_ret, relevant, 3)
    ir3 = recall_at_k(imp_ret, relevant, 3)
    irr = reciprocal_rank(imp_ret, relevant)
    inn = ndcg_at_k(imp_ret, relevant, 5)
    
    sq = query[:38]
    print(f"{sq:<40} | {op3:>6.3f} {or3:>6.3f} {orr:>6.3f} {on:>6.3f} | {ip3:>6.3f} {ir3:>6.3f} {irr:>6.3f} {inn:>6.3f}")

                                         |          Original           |          Improved          
Query                                    |    P@3    R@3    MRR   NDCG |    P@3    R@3    MRR   NDCG
How do I bake cookies?                   |  0.667  0.667  1.000  0.922 |  1.000  1.000  1.000  1.000
How should I cook pasta?                 |  0.667  1.000  0.500  0.670 |  0.667  1.000  1.000  1.000
Tell me about working with chocolate     |  1.000  1.000  1.000  1.000 |  1.000  1.000  1.000  1.000


The improved retriever boosts the pasta query significantly across all metrics (MRR goes from 0.5 to 1.0) and pushes the cookie query's precision from 0.667 to 1.0 by getting all three relevant docs into the top 3. The chocolate query was already near perfect, so it stays the same.

## Summary

You have now built four core RAG evaluation metrics from scratch and observed exactly how each one works:

| Metric | Question It Answers | Key Insight |
|--------|--------------------|------------|
| **Precision@K** | How many top K results are relevant? | Good for measuring noise |
| **Recall@K** | How many relevant docs did you find? | Good for measuring coverage |
| **MRR** | How quickly do you find the first relevant doc? | Critical for single answer RAG |
| **NDCG@K** | How good is the full ranking quality? | The most complete metric |

The most important takeaway: **no single metric tells the whole story.** In practice, track at least two: one that measures precision (are you returning noise?) and one that measures ranking quality (are the best docs on top?). For RAG specifically, MRR and NDCG@5 are a strong pair.

To evaluate your own RAG system, you need three things: a set of test queries, ground truth relevance labels, and these metric functions. The hardest part is creating good ground truth, but even a small set of 20 to 30 labeled queries can reveal major retrieval problems.